<a href="https://colab.research.google.com/github/VasilisPapageorgiou/Amortization-of-Risk-Indicators/blob/main/Amortization_Exp5_3_C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =====================================================================================
# EXPERIMENT 5.3-C — ACCURACY-CONDITIONAL AMORTIZATION
# FINAL / JASA-LEVEL / RESUMABLE / CPU-ONLY
#
# R = 500, 1000, 2000, 5000, 10000
# 5 independent neural-training replications
#
# Main question:
#
#   Given required tail-risk accuracy epsilon,
#   how many downstream evaluations are required before the
#   exact-teacher neural emulator amortizes its offline cost?
#
#   R_epsilon = min{R : median E_rho(R) <= epsilon}
#
#   B*_epsilon(N)
#       = [C_teach(R_epsilon) + C_train(R_epsilon)]
#         / [c_E(N) - c_F(N)].
#
# IMPORTANT:
#   C_teach is estimated from controlled exact-query benchmarks,
#   NOT by summing concurrent target-generation timings.
#
# Main outputs:
#   main_table_5_3C.tex
#   figure_5_3C_main.pdf
#
# Supplementary diagnostics:
#   accuracy_full.csv
#   runtime_full.csv
#   break_even_full.csv
#   threshold_selection.csv
#   training_diagnostics.csv
#
# CPU ONLY | SPARSE LU | NO MATRIX INVERSE
# =====================================================================================


# =====================================================================================
# 0. DRIVE + ENVIRONMENT
# =====================================================================================

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os
os.environ["OMP_NUM_THREADS"]="1"
os.environ["OPENBLAS_NUM_THREADS"]="1"
os.environ["MKL_NUM_THREADS"]="1"
os.environ["NUMEXPR_NUM_THREADS"]="1"


# =====================================================================================
# 1. IMPORTS
# =====================================================================================

import json, hashlib, math, pickle, random, time, platform
from dataclasses import dataclass, asdict
from datetime import datetime
from functools import lru_cache
from pathlib import Path
from typing import Tuple

import numpy as np
import pandas as pd

from scipy import sparse
from scipy.sparse.linalg import splu
from scipy.stats import qmc
from joblib import Parallel, delayed

import torch
import torch.nn as nn
import matplotlib.pyplot as plt


# =====================================================================================
# 2. CONFIG
# =====================================================================================

@dataclass
class Config:
    seed:int = 20260820

    beta:Tuple[float,float] = (.30,1.50)
    gamma:Tuple[float,float] = (.20,1.00)
    omega:Tuple[float,float] = (.02,.50)
    frac:Tuple[float,float] = (.02,.20)

    trainN:Tuple[int,...] = tuple(range(40,401,20))

    width:int = 128
    depth:int = 3
    batch:int = 64
    epochs:int = 500

    lr:float = 1e-3
    wd:float = 1e-6

    patience:int = 20
    delta:float = 1e-6
    clip:float = 5.

    lambda_rho:float = 1.
    lambda_tau:float = .02

    prob_tol:float = 1e-10
    var_tol:float = 1e-8
    refine:int = 3


cfg=Config()

R_GRID=(500,1000,2000,5000,10000)
RMAX=max(R_GRID)

N_REP=5

VAL_PER_N=20
TEST_N=tuple(range(40,401,10))
TEST_PER_N=10

N_VAL=VAL_PER_N*len(cfg.trainN)
N_TEST=TEST_PER_N*len(TEST_N)

EPS=(.05,.025,.01)

# Deployment population sizes for amortization curves
BENCH_N=(50,75,100,125,150,175,200,225,250,275,300,325,350,375,400)

# Runtime also needed at every training N to estimate C_teach.
BENCH_ALL_N=tuple(sorted(set(BENCH_N)|set(cfg.trainN)))

# More independent configurations; fewer technical repeats.
BENCH_PER_N=10
EXACT_REPEATS=3
NEURAL_BLOCKS=5
NEURAL_REPEATS=100

BOOT_B=3000

EXACT_CHUNK=50
CHECKPOINT_EVERY=5

N_SCALE=max(cfg.trainN)

assert max(R_GRID)==RMAX


# =====================================================================================
# 3. RESUME / START-NEW PROTECTION
# =====================================================================================

CODE_VERSION="5.3C_JASA_v1"

SCIENTIFIC_CONFIG={
    "code_version":CODE_VERSION,
    "config":asdict(cfg),
    "R_GRID":R_GRID,
    "N_REP":N_REP,
    "VAL_PER_N":VAL_PER_N,
    "TEST_N":TEST_N,
    "TEST_PER_N":TEST_PER_N,
    "EPS":EPS,
    "BENCH_N":BENCH_N,
    "BENCH_ALL_N":BENCH_ALL_N,
    "BENCH_PER_N":BENCH_PER_N,
    "EXACT_REPEATS":EXACT_REPEATS,
    "NEURAL_BLOCKS":NEURAL_BLOCKS,
    "NEURAL_REPEATS":NEURAL_REPEATS,
    "protocol":"balanced nested exact prefixes; controlled query-cost amortization"
}

def signature(x):
    s=json.dumps(x,sort_keys=True,default=str)
    return hashlib.sha256(s.encode()).hexdigest()[:16]

SIG=signature(SCIENTIFIC_CONFIG)

BASE=Path(
    "/content/drive/MyDrive/StatisticalLearning/"
    "Experiment_5_3C_JASA"
)
BASE.mkdir(parents=True,exist_ok=True)

print("\n"+"="*82)
print("1 = RESUME latest run")
print("2 = START NEW run")
print("="*82)

mode=input("Choose 1 or 2: ").strip()

if mode not in ("1","2"):
    raise RuntimeError("Choose 1 or 2.")

latest=BASE/"latest_run.txt"

if mode=="2":
    stamp=datetime.now().strftime("%Y%m%d_%H%M%S")
    ROOT=BASE/f"run_{stamp}"
    ROOT.mkdir()

    with open(ROOT/"manifest.pkl","wb") as f:
        pickle.dump(
            {"signature":SIG,"scientific_config":SCIENTIFIC_CONFIG},
            f,pickle.HIGHEST_PROTOCOL
        )

    latest.write_text(ROOT.name)

else:
    if not latest.exists():
        raise RuntimeError("No previous run exists. Choose START NEW.")

    ROOT=BASE/latest.read_text().strip()

    with open(ROOT/"manifest.pkl","rb") as f:
        old=pickle.load(f)

    if old["signature"]!=SIG:
        raise RuntimeError(
            "Scientific specification changed. "
            "RESUME aborted; choose START NEW."
        )


CACHE=ROOT/"cache"
EXACT=CACHE/"exact"
MODELS=CACHE/"models"
CKPT=CACHE/"checkpoints"
RUNTIME_CACHE=CACHE/"runtime"
OUT=ROOT/"results"

for d in (CACHE,EXACT,MODELS,CKPT,RUNTIME_CACHE,OUT):
    d.mkdir(parents=True,exist_ok=True)

print("ROOT:",ROOT)
print("Signature:",SIG)


# =====================================================================================
# 4. UTILITIES / CPU
# =====================================================================================

def atomic_pickle(x,path):
    path=Path(path)
    tmp=path.with_suffix(path.suffix+".tmp")
    with open(tmp,"wb") as f:
        pickle.dump(x,f,pickle.HIGHEST_PROTOCOL)
    os.replace(tmp,path)

def atomic_torch(x,path):
    path=Path(path)
    tmp=path.with_suffix(path.suffix+".tmp")
    torch.save(x,tmp)
    os.replace(tmp,path)

def load_pickle(path,default=None):
    try:
        with open(path,"rb") as f:
            return pickle.load(f)
    except Exception:
        return default

def seed_all(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)

seed_all(cfg.seed)

CPU=os.cpu_count() or 1
N_EXACT=max(1,min(2,CPU))
TORCH_THREADS=max(1,min(8,CPU))

torch.set_num_threads(TORCH_THREADS)

try:
    torch.set_num_interop_threads(1)
except RuntimeError:
    pass

try:
    torch.use_deterministic_algorithms(True,warn_only=True)
except Exception:
    pass

device=torch.device("cpu")

print(
    f"CPU={CPU} | exact workers={N_EXACT} | "
    f"Torch threads={torch.get_num_threads()}"
)

with open(OUT/"computational_environment.json","w") as f:
    json.dump(
        {
            "platform":platform.platform(),
            "python":platform.python_version(),
            "torch":torch.__version__,
            "numpy":np.__version__,
            "cpu_count":CPU,
            "exact_workers":N_EXACT,
            "torch_threads":TORCH_THREADS
        },
        f,indent=2
    )


# =====================================================================================
# 5. RECORD + SIRS TOPOLOGY
# =====================================================================================

@dataclass
class Rec:
    b:float
    g:float
    w:float
    N:int
    i0:int
    p:np.ndarray
    mt:float
    vt:float
    tv:bool


@lru_cache(None)
def topo(N):
    states=[
        (s,i)
        for i in range(1,N+1)
        for s in range(N-i+1)
    ]

    ix={x:j for j,x in enumerate(states)}
    M=len(states)

    ir=[]; ic=[]; ib=[]
    rr=[]; rc=[]; rb=[]
    wr=[]; wc=[]; wb=[]

    db=np.zeros(M)
    dg=np.zeros(M)
    dw=np.zeros(M)
    qb=np.zeros(M)

    for j,(s,i) in enumerate(states):
        r=N-s-i

        if s:
            ir.append(j)
            ic.append(ix[(s-1,i+1)])
            rate=s*i/N
            ib.append(rate)
            db[j]=rate

        dg[j]=i

        if i==1:
            qb[j]=i
        else:
            rr.append(j)
            rc.append(ix[(s,i-1)])
            rb.append(i)

        if r:
            wr.append(j)
            wc.append(ix[(s+1,i)])
            wb.append(r)
            dw[j]=r

    A=lambda x,d=float:np.asarray(x,dtype=d)

    return (
        ix,M,
        A(ir,int),A(ic,int),A(ib),
        A(rr,int),A(rc,int),A(rb),
        A(wr,int),A(wc,int),A(wb),
        db,dg,dw,qb
    )


# =====================================================================================
# 6. NUMERICALLY REFINED SPARSE SOLVES
# =====================================================================================

def refined(A,lu,b,transpose=False):
    b=np.asarray(b,dtype=np.float64)
    mode="T" if transpose else "N"

    x=lu.solve(b,trans=mode)

    for _ in range(cfg.refine):
        r=b-(A.T@x if transpose else A@x)

        if not np.all(np.isfinite(r)):
            break

        rel=np.linalg.norm(r,np.inf)/max(np.linalg.norm(b,np.inf),1.)

        if rel<1e-11:
            break

        x+=lu.solve(r,trans=mode)

    return np.asarray(x,dtype=np.float64)


def moment_factor(A,ordering):
    d=np.abs(A.diagonal())
    scale=1./np.maximum(d,np.finfo(float).tiny)

    As=(
        sparse.diags(scale)
        @A
    ).tocsc()

    return As,splu(As,permc_spec=ordering),scale


def moment_solve(A,As,lu,scale,b):
    # Solve A x = b via diag(scale) A x = diag(scale)b.
    rhs=scale*np.asarray(b,dtype=np.float64)
    x=lu.solve(rhs)

    for _ in range(cfg.refine):
        r=np.asarray(b,dtype=np.float64)-A@x

        if not np.all(np.isfinite(r)):
            break

        rel=np.linalg.norm(r,np.inf)/max(
            np.linalg.norm(b,np.inf),1.
        )

        if rel<1e-11:
            break

        x+=lu.solve(scale*r)

    return np.asarray(x,dtype=np.float64)


# =====================================================================================
# 7. COMPLETE EXACT QUERY
# =====================================================================================

def exact_full(b,g,w,N,i0):
    (
        ix,M,ir,ic,ib,rr,rc,rb,
        wr,wc,wb,db,dg,dw,qb
    )=topo(N)

    rows=np.r_[ir,rr,wr,np.arange(M)]
    cols=np.r_[ic,rc,wc,np.arange(M)]

    vals=np.r_[
        b*ib,
        g*rb,
        w*wb,
        -(b*db+g*dg+w*dw)
    ]

    T=sparse.coo_matrix(
        (vals,(rows,cols)),
        shape=(M,M),
        dtype=np.float64
    ).tocsc()

    D1=sparse.coo_matrix(
        (b*ib,(ir,ic)),
        shape=(M,M),
        dtype=np.float64
    ).tocsc()

    D0=(T-D1).tocsc()
    q=g*qb
    initial=ix[(N-i0,i0)]

    # -------------------------------------------------------------------------
    # Count distribution
    # -------------------------------------------------------------------------

    A0=(-D0).tocsc()
    lu0=splu(A0,permc_spec="COLAMD")

    bvec=refined(A0,lu0,q)

    v=np.zeros(M)
    v[initial]=1.

    D1T=D1.T.tocsr()
    p=np.zeros(N+2)

    for k in range(N+1):
        p[k]=v@bvec

        y=refined(
            A0,lu0,v,
            transpose=True
        )

        v=np.asarray(
            D1T@y
        ).ravel()

    p[-1]=v.sum()
    p[np.abs(p)<cfg.prob_tol]=0.

    if (
        not np.all(np.isfinite(p))
        or p.min() < -cfg.prob_tol
    ):
        raise RuntimeError("Invalid exact PMF.")

    p=np.maximum(p,0.)
    mass=p.sum()

    if not np.isfinite(mass) or abs(mass-1.)>1e-5:
        raise RuntimeError(f"Invalid exact mass={mass}.")

    p/=mass

    rho=np.flip(
        np.cumsum(
            np.flip(p[1:])
        )
    )

    # -------------------------------------------------------------------------
    # Extinction moments
    # -------------------------------------------------------------------------

    A=(-T).tocsc()

    for ordering in ("COLAMD","MMD_AT_PLUS_A"):
        try:
            As,lu,scale=moment_factor(A,ordering)

            m1=moment_solve(
                A,As,lu,scale,
                np.ones(M)
            )

            mean=float(m1[initial])

            if (
                not np.all(np.isfinite(m1))
                or mean<=0
            ):
                raise ArithmeticError("Invalid E(tau).")

            m2=moment_solve(
                A,As,lu,scale,
                2.*m1
            )

            second=float(m2[initial])

            if (
                not np.all(np.isfinite(m2))
                or second<=0
            ):
                raise ArithmeticError("Invalid E(tau^2).")

            raw=(
                np.longdouble(second)
                -
                np.longdouble(mean)**2
            )

            tol=cfg.var_tol*max(
                abs(second),
                mean*mean,
                1.
            )

            if np.isfinite(raw) and raw>=-tol:
                var=max(float(raw),0.)

            else:
                C=T.tocoo()
                off=C.row!=C.col

                source=np.bincount(
                    C.row[off],
                    weights=(
                        C.data[off]
                        *
                        (
                            m1[C.col[off]]
                            -
                            m1[C.row[off]]
                        )**2
                    ),
                    minlength=M
                ).astype(float)

                source+=q*m1*m1

                if (
                    not np.all(np.isfinite(source))
                    or source.min() < -1e-8
                ):
                    raise ArithmeticError("Invalid variance RHS.")

                vv=moment_solve(
                    A,As,lu,scale,
                    np.maximum(source,0.)
                )

                var=float(vv[initial])

            if not np.isfinite(var) or var<0:
                raise ArithmeticError("Invalid Var(tau).")

            return p,rho,mean,var,True

        except Exception:
            pass

    return p,rho,np.nan,np.nan,False


# =====================================================================================
# 8. DESIGN
# =====================================================================================

def design(n,Ns,seed):
    U=qmc.LatinHypercube(
        d=4,
        seed=seed
    ).random(n)

    scale=lambda x,a:a[0]+(a[1]-a[0])*x

    b=scale(U[:,0],cfg.beta)
    g=scale(U[:,1],cfg.gamma)
    w=scale(U[:,2],cfg.omega)
    f=scale(U[:,3],cfg.frac)

    Nv=np.tile(
        np.asarray(Ns),
        math.ceil(n/len(Ns))
    )[:n]

    rng=np.random.default_rng(seed+99)
    rng.shuffle(Nv)

    i0=np.asarray([
        int(np.clip(
            round(f[j]*Nv[j]),
            2,Nv[j]
        ))
        for j in range(n)
    ])

    for N in Ns:
        z=np.where(Nv==N)[0]

        if len(z):
            k=max(1,round(.25*len(z)))
            i0[
                rng.choice(z,k,replace=False)
            ]=1

    return [
        (
            float(b[j]),
            float(g[j]),
            float(w[j]),
            int(Nv[j]),
            int(i0[j])
        )
        for j in range(n)
    ]


def balanced_config_order(configs,Ns,seed):
    rng=np.random.default_rng(seed)

    groups={}

    for N in Ns:
        for flag in (0,1):
            z=np.asarray([
                j for j,x in enumerate(configs)
                if x[3]==N and int(x[4]==1)==flag
            ],dtype=int)

            rng.shuffle(z)
            groups[(N,flag)]=list(z)

    ptr={k:0 for k in groups}
    used={N:0 for N in Ns}
    used1=0
    order=[]

    for pos in range(len(configs)):
        preferred=1 if used1<.25*(pos+1) else 0
        chosen=None

        for flag in (preferred,1-preferred):
            cand=[
                N for N in Ns
                if ptr[(N,flag)]<len(groups[(N,flag)])
            ]

            if cand:
                m=min(used[N] for N in cand)
                cand=[N for N in cand if used[N]==m]
                chosen=(int(rng.choice(cand)),flag)
                break

        if chosen is None:
            raise RuntimeError("Balanced ordering failed.")

        N,flag=chosen
        j=groups[(N,flag)][ptr[(N,flag)]]

        ptr[(N,flag)]+=1
        used[N]+=1
        used1+=flag

        order.append(j)

    return np.asarray(order,dtype=int)


# =====================================================================================
# 9. RESUMABLE EXACT DATA
# =====================================================================================

def records_match(records,configs,tol=1e-12):
    if records is None or len(records)!=len(configs):
        return False

    for r,x in zip(records,configs):
        b,g,w,N,i0=x

        if (
            abs(r.b-b)>tol
            or abs(r.g-g)>tol
            or abs(r.w-w)>tol
            or r.N!=N
            or r.i0!=i0
        ):
            return False

    return True


def exact_one(j,x):
    p,_,m,v,ok=exact_full(*x)
    return j,Rec(*x,p,m,v,ok)


def exact_set(configs,name):
    full=EXACT/f"{name}_full.pkl"

    if full.exists():
        z=load_pickle(full)

        if records_match(z,configs):
            print(f"{name}: full cache ({len(z):,})")
            return z

    folder=EXACT/name
    folder.mkdir(parents=True,exist_ok=True)

    ans=[]

    for start in range(0,len(configs),EXACT_CHUNK):
        end=min(start+EXACT_CHUNK,len(configs))
        cc=configs[start:end]

        f=folder/f"chunk_{start:05d}_{end:05d}.pkl"
        part=load_pickle(f) if f.exists() else None

        if not records_match(part,cc):
            jobs=list(enumerate(cc))
            jobs.sort(
                key=lambda z:z[1][3],
                reverse=True
            )

            t0=time.perf_counter()

            out=Parallel(
                n_jobs=N_EXACT,
                backend="threading"
            )(
                delayed(exact_one)(j,x)
                for j,x in jobs
            )

            out.sort(key=lambda z:z[0])
            part=[r for _,r in out]

            atomic_pickle(part,f)

            print(
                f"{name} {start:5d}:{end:5d} | "
                f"{time.perf_counter()-t0:.1f}s"
            )

        else:
            print(
                f"{name} {start:5d}:{end:5d} | cache"
            )

        ans.extend(part)

    if not records_match(ans,configs):
        raise RuntimeError(f"{name}: cache/design mismatch.")

    atomic_pickle(ans,full)
    return ans


# =====================================================================================
# 10. BALANCED NESTED TRAINING DESIGN + TEST DATA
# =====================================================================================

base_train_design=design(
    RMAX,
    cfg.trainN,
    cfg.seed+1
)

order=balanced_config_order(
    base_train_design,
    cfg.trainN,
    cfg.seed+500
)

train_design=[
    base_train_design[int(j)]
    for j in order
]

val_design=design(
    N_VAL,
    cfg.trainN,
    cfg.seed+2
)

test_design=design(
    N_TEST,
    TEST_N,
    cfg.seed+3
)

train=exact_set(train_design,"TRAIN")
val=exact_set(val_design,"VALIDATION")
test=exact_set(test_design,"TEST")

print(
    f"\nTrain={len(train):,} | "
    f"Val={len(val):,} | Test={len(test):,}"
)

for R in R_GRID:
    x=train[:R]
    counts={
        N:sum(r.N==N for r in x)
        for N in cfg.trainN
    }

    print(
        f"R={R:5d} | "
        f"N count range={min(counts.values())}-{max(counts.values())} | "
        f"i0=1={sum(r.i0==1 for r in x)/R:.1%}"
    )


# =====================================================================================
# 11. NETWORKS / PACKING
# =====================================================================================

def mlp(din,dout):
    L=[]
    d=din

    for _ in range(cfg.depth):
        L += [
            nn.Linear(d,cfg.width),
            nn.SiLU()
        ]
        d=cfg.width

    L.append(nn.Linear(d,dout))
    return nn.Sequential(*L)


class HazardNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=mlp(6,1)

    def forward(self,x):
        return torch.sigmoid(
            self.net(x).squeeze(-1)
        )


class TauNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=mlp(5,2)

    def forward(self,x):
        return torch.nn.functional.softplus(
            self.net(x)
        )


def pack(records):
    groups={}

    for j,r in enumerate(records):
        groups.setdefault(r.N,[]).append(j)

    P={}

    for N,idx in groups.items():
        idx=np.asarray(idx,dtype=int)
        rr=[records[j] for j in idx]

        B=len(rr)
        K=N+1

        b=torch.tensor([r.b for r in rr],dtype=torch.float32)[:,None]
        g=torch.tensor([r.g for r in rr],dtype=torch.float32)[:,None]
        w=torch.tensor([r.w for r in rr],dtype=torch.float32)[:,None]
        ns=torch.full((B,1),N/N_SCALE,dtype=torch.float32)
        i0=torch.tensor([r.i0/N for r in rr],dtype=torch.float32)[:,None]
        c=(torch.arange(K,dtype=torch.float32)/N)[None,:]

        Xh=torch.stack([
            b.expand(B,K),
            g.expand(B,K),
            w.expand(B,K),
            ns.expand(B,K),
            i0.expand(B,K),
            c.expand(B,K)
        ],dim=2).contiguous()

        Yp=torch.from_numpy(
            np.stack([r.p for r in rr]).astype(np.float32)
        )

        Xt=torch.column_stack([
            b[:,0],g[:,0],w[:,0],ns[:,0],i0[:,0]
        ])

        mask=torch.tensor(
            [r.tv for r in rr],
            dtype=torch.bool
        )

        Yt=np.zeros((B,2),dtype=np.float32)

        for j,r in enumerate(rr):
            if r.tv:
                Yt[j]=np.log1p([
                    r.mt,
                    r.vt
                ])

        P[N]={
            "Xh":Xh,
            "Yp":Yp,
            "Xt":Xt,
            "Yt":torch.from_numpy(Yt),
            "mask":mask,
            "orig":idx,
            "n":B
        }

    return P


def reconstruct(h):
    B=h.shape[0]

    before=torch.cat([
        torch.ones((B,1),dtype=h.dtype),
        torch.cumprod(
            1-h[:,:-1],
            dim=1
        )
    ],dim=1)

    return torch.cat([
        before*h,
        torch.prod(
            1-h,
            dim=1,
            keepdim=True
        )
    ],dim=1)


def tail(p):
    return torch.flip(
        torch.cumsum(
            torch.flip(
                p[:,1:],
                dims=[1]
            ),
            dim=1
        ),
        dims=[1]
    )


def predict_p(hnet,X):
    B,K,_=X.shape

    h=hnet(
        X.reshape(B*K,6)
    ).reshape(B,K)

    return reconstruct(h)


def batch_loss(hnet,tnet,G,idx):
    X=G["Xh"][idx]
    Y=G["Yp"][idx]

    P=predict_p(hnet,X)

    Lp=torch.sum(
        (P-Y)**2,
        dim=1
    ).mean()

    Lrho=torch.mean(
        (tail(P)-tail(Y))**2,
        dim=1
    ).mean()

    valid=G["mask"][idx]

    if torch.any(valid):
        target=G["Yt"][idx][valid]
        pred=tnet(G["Xt"][idx][valid])

        Ltau=torch.sum(
            (pred-target)**2
            /
            (1+target*target),
            dim=1
        ).mean()
    else:
        Ltau=torch.zeros(())

    return (
        Lp
        +
        cfg.lambda_rho*Lrho
        +
        cfg.lambda_tau*Ltau
    )


def batches(P,rng,shuffle=True):
    ans=[]

    for N,G in P.items():
        idx=np.arange(G["n"])

        if shuffle:
            rng.shuffle(idx)

        for s in range(0,len(idx),cfg.batch):
            ans.append(
                (N,idx[s:s+cfg.batch])
            )

    if shuffle:
        rng.shuffle(ans)

    return ans


@torch.no_grad()
def validation(h,t,P):
    h.eval()
    t.eval()

    total=0.
    n=0
    rng=np.random.default_rng(1)

    for N,idx in batches(P,rng,False):
        L=batch_loss(h,t,P[N],idx)

        total+=L.item()*len(idx)
        n+=len(idx)

    return total/n


VAL=pack(val)
TEST=pack(test)


# =====================================================================================
# 12. RESUMABLE TRAINING
# =====================================================================================

def cpu_state(state):
    return {
        k:v.detach().cpu()
        for k,v in state.items()
    }


def fit(TR,rep,R):
    name=f"rep{rep:02d}_R{R}"

    final=MODELS/f"{name}_FINAL.pt"
    checkpoint=CKPT/f"{name}_CHECKPOINT.pt"

    h=HazardNet()
    t=TauNet()

    if final.exists():
        z=torch.load(
            final,
            map_location="cpu",
            weights_only=False
        )

        h.load_state_dict(z["h"])
        t.load_state_dict(z["t"])

        return h,t,z

    # Same initialization across R within replication.
    seed=cfg.seed+7001+rep*100003
    seed_all(seed)

    h=HazardNet()
    t=TauNet()

    pars=list(h.parameters())+list(t.parameters())

    opt=torch.optim.AdamW(
        pars,
        lr=cfg.lr,
        weight_decay=cfg.wd
    )

    sch=torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt,
        factor=.5,
        patience=15
    )

    rng=np.random.default_rng(seed+77)

    start_epoch=1
    best=np.inf
    best_epoch=0
    best_h=None
    best_t=None
    wait=0
    history=[]
    previous_elapsed=0.

    if checkpoint.exists():
        z=torch.load(
            checkpoint,
            map_location="cpu",
            weights_only=False
        )

        h.load_state_dict(z["h_current"])
        t.load_state_dict(z["t_current"])

        opt.load_state_dict(z["optimizer"])
        sch.load_state_dict(z["scheduler"])

        start_epoch=z["epoch"]+1

        best=z["best"]
        best_epoch=z["best_epoch"]
        best_h=z["best_h"]
        best_t=z["best_t"]
        wait=z["wait"]

        history=z["history"]
        previous_elapsed=z["elapsed"]

        rng.bit_generator.state=z["rng_state"]
        torch.set_rng_state(z["torch_rng"])

        print(
            f"{name}: resume epoch {start_epoch}"
        )

    start=time.perf_counter()
    stopped=cfg.epochs

    for epoch in range(start_epoch,cfg.epochs+1):
        h.train()
        t.train()

        for N,idx in batches(TR,rng,True):
            opt.zero_grad(set_to_none=True)

            L=batch_loss(
                h,t,TR[N],idx
            )

            if not torch.isfinite(L):
                raise RuntimeError(
                    f"{name}: non-finite loss."
                )

            L.backward()

            torch.nn.utils.clip_grad_norm_(
                pars,
                cfg.clip
            )

            opt.step()

        v=validation(h,t,VAL)
        sch.step(v)

        history.append(float(v))

        if best_h is None or v<best-cfg.delta:
            best=float(v)
            best_epoch=epoch
            best_h=cpu_state(h.state_dict())
            best_t=cpu_state(t.state_dict())
            wait=0
        else:
            wait+=1

        if (
            epoch==1
            or epoch%CHECKPOINT_EVERY==0
        ):
            elapsed=(
                previous_elapsed
                +time.perf_counter()
                -start
            )

            atomic_torch({
                "epoch":epoch,
                "h_current":cpu_state(h.state_dict()),
                "t_current":cpu_state(t.state_dict()),
                "optimizer":opt.state_dict(),
                "scheduler":sch.state_dict(),
                "best":best,
                "best_epoch":best_epoch,
                "best_h":best_h,
                "best_t":best_t,
                "wait":wait,
                "history":history,
                "rng_state":rng.bit_generator.state,
                "torch_rng":torch.get_rng_state(),
                "elapsed":elapsed
            },checkpoint)

            print(
                f"{name} | ep={epoch:3d} | "
                f"val={v:.4e} | best={best:.4e} | "
                f"wait={wait}"
            )

        if wait>=cfg.patience:
            stopped=epoch
            break

    elapsed=(
        previous_elapsed
        +time.perf_counter()
        -start
    )

    if best_h is None or best_t is None:
        raise RuntimeError(
            f"{name}: no valid best model."
        )

    h.load_state_dict(best_h)
    t.load_state_dict(best_t)

    payload={
        "h":best_h,
        "t":best_t,
        "best_epoch":best_epoch,
        "stopped_epoch":stopped,
        "validation":best,
        "training_sec":elapsed,
        "history":history
    }

    atomic_torch(payload,final)

    if checkpoint.exists():
        checkpoint.unlink()

    return h,t,payload


# =====================================================================================
# 13. COMPLETE TEST METRICS
# =====================================================================================

@torch.no_grad()
def evaluate(h,t,P,records):
    h.eval()
    t.eval()

    n=len(records)

    out={
        k:np.full(n,np.nan)
        for k in (
            "E2",
            "Erho",
            "mean_tau_rel",
            "var_tau_rel"
        )
    }

    for N,G in P.items():
        for s in range(0,G["n"],cfg.batch):
            e=min(s+cfg.batch,G["n"])

            loc=np.arange(s,e)
            orig=G["orig"][loc]

            X=G["Xh"][loc]
            Y=G["Yp"][loc]

            Ph=predict_p(h,X)

            out["E2"][orig]=(
                torch.linalg.vector_norm(
                    Ph-Y,
                    dim=1
                ).numpy()
            )

            out["Erho"][orig]=(
                torch.max(
                    torch.abs(
                        tail(Ph)-tail(Y)
                    ),
                    dim=1
                ).values.numpy()
            )

            mask=G["mask"][loc]

            if torch.any(mask):
                z=(
                    t(G["Xt"][loc][mask])
                    .numpy()
                    .astype(np.float64)
                )

                moments=np.expm1(
                    np.clip(z,0.,80.)
                )

                valid_orig=orig[
                    mask.numpy()
                ]

                for q,j in enumerate(valid_orig):
                    r=records[int(j)]

                    out["mean_tau_rel"][j]=(
                        abs(moments[q,0]-r.mt)
                        /
                        r.mt
                    )

                    out["var_tau_rel"][j]=(
                        abs(moments[q,1]-r.vt)
                        /
                        max(r.vt,1e-300)
                    )

    return out


# =====================================================================================
# 14. ACCURACY EXPERIMENT
# =====================================================================================

PROGRESS_FILE=ROOT/"accuracy_progress.pkl"

progress=load_pickle(
    PROGRESS_FILE,
    {"results":{}}
)

progress.setdefault("results",{})

for rep in range(1,N_REP+1):
    print("\n"+"="*90)
    print(f"TRAINING REPLICATION {rep}/{N_REP}")
    print("="*90)

    for R in R_GRID:
        key=f"rep{rep:02d}_R{R}"

        if key in progress["results"]:
            print(key,"| complete")
            continue

        TR=pack(train[:R])

        h,t,meta=fit(
            TR,rep,R
        )

        z=evaluate(
            h,t,TEST,test
        )

        progress["results"][key]={
            "rep":rep,
            "R":R,
            **z,
            "training_sec":meta["training_sec"],
            "best_epoch":meta["best_epoch"],
            "stopped_epoch":meta["stopped_epoch"]
        }

        atomic_pickle(
            progress,
            PROGRESS_FILE
        )

        print(
            f"R={R:5d} | "
            f"E2={np.nanmedian(z['E2']):.4g} | "
            f"Erho={np.nanmedian(z['Erho']):.4g}"
        )

        del TR,h,t


# =====================================================================================
# 15. HIERARCHICAL ACCURACY SUMMARIES
# =====================================================================================

def accuracy_matrix(R,metric):
    return np.vstack([
        progress["results"][
            f"rep{rep:02d}_R{R}"
        ][metric]
        for rep in range(1,N_REP+1)
    ])


def hier_ci(A,B=BOOT_B,seed=1):
    A=np.asarray(A,float)
    nr,nc=A.shape

    rng=np.random.default_rng(seed)
    z=[]

    for _ in range(B):
        ir=rng.integers(0,nr,size=nr)
        ic=rng.integers(0,nc,size=nc)

        v=np.nanmedian(
            A[ir][:,ic]
        )

        if np.isfinite(v):
            z.append(v)

    z=np.asarray(z)

    return (
        float(np.quantile(z,.025)),
        float(np.quantile(z,.975))
    )


accuracy_rows=[]

for R in R_GRID:
    for j,metric in enumerate(
        ("E2","Erho","mean_tau_rel","var_tau_rel")
    ):
        A=accuracy_matrix(R,metric)
        x=A[np.isfinite(A)]

        lo,hi=hier_ci(
            A,
            seed=cfg.seed+R+j
        )

        accuracy_rows.append({
            "R":R,
            "metric":metric,
            "median":float(np.median(x)),
            "q1":float(np.quantile(x,.25)),
            "q3":float(np.quantile(x,.75)),
            "p95":float(np.quantile(x,.95)),
            "CI_low":lo,
            "CI_high":hi
        })


accuracy=pd.DataFrame(
    accuracy_rows
)

accuracy.to_csv(
    OUT/"accuracy_full.csv",
    index=False
)


def A(R,metric):
    return accuracy[
        (accuracy.R==R)
        &
        (accuracy.metric==metric)
    ].iloc[0]


# =====================================================================================
# 16. TRAINING COST SUMMARY
# =====================================================================================

training_rows=[]

for rep in range(1,N_REP+1):
    for R in R_GRID:
        z=progress["results"][
            f"rep{rep:02d}_R{R}"
        ]

        training_rows.append({
            "rep":rep,
            "R":R,
            "training_sec":z["training_sec"],
            "best_epoch":z["best_epoch"],
            "stopped_epoch":z["stopped_epoch"]
        })


training=pd.DataFrame(training_rows)

training.to_csv(
    OUT/"training_diagnostics.csv",
    index=False
)


# =====================================================================================
# 17. LOAD ONE RMAX MODEL FOR QUERY-TIME BENCHMARK
#
# Architecture is identical for all R; parameter values do not materially change
# the forward-pass computational graph.
# =====================================================================================

runtime_model=MODELS/"rep01_R10000_FINAL.pt"

if not runtime_model.exists():
    raise RuntimeError(
        "R=10000 replication-1 model is required."
    )

z=torch.load(
    runtime_model,
    map_location="cpu",
    weights_only=False
)

hmax=HazardNet()
tmax=TauNet()

hmax.load_state_dict(z["h"])
tmax.load_state_dict(z["t"])

hmax.eval()
tmax.eval()


# =====================================================================================
# 18. COMPLETE NEURAL QUERY
# =====================================================================================

@torch.inference_mode()
def neural_query(h,t,x):
    b,g,w,N,i0=x
    K=N+1

    c=torch.arange(
        K,
        dtype=torch.float32
    )/N

    X=torch.column_stack([
        torch.full((K,),b),
        torch.full((K,),g),
        torch.full((K,),w),
        torch.full((K,),N/N_SCALE),
        torch.full((K,),i0/N),
        c
    ])

    H=h(X).reshape(1,K)
    P=reconstruct(H)[0]
    RHO=tail(P.unsqueeze(0))[0]

    Xt=torch.tensor(
        [[b,g,w,N/N_SCALE,i0/N]],
        dtype=torch.float32
    )

    moments=torch.expm1(
        torch.clamp(
            t(Xt)[0],
            0,
            80
        )
    )

    return (
        P,
        RHO,
        moments[0],
        moments[1]
    )


# =====================================================================================
# 19. CONTROLLED QUERY-TIME BENCHMARK
# =====================================================================================

bench_design=design(
    BENCH_PER_N*len(BENCH_ALL_N),
    BENCH_ALL_N,
    cfg.seed+50
)


def speedup_boot(exact_times,neural_times,B=BOOT_B,seed=1):
    e=np.asarray(exact_times,float)
    f=np.asarray(neural_times,float)

    rng=np.random.default_rng(seed)
    z=np.empty(B)

    for b in range(B):
        idx=rng.integers(
            0,len(e),
            size=len(e)
        )

        z[b]=(
            np.median(e[idx])
            /
            np.median(f[idx])
        )

    return (
        float(np.quantile(z,.025)),
        float(np.quantile(z,.975))
    )


runtime_detail={}
runtime_rows=[]

print("\n"+"="*100)
print("CONTROLLED COMPLETE-QUERY BENCHMARK")
print("="*100)

for N in BENCH_ALL_N:
    xs=[
        x for x in bench_design
        if x[3]==N
    ]

    if len(xs)!=BENCH_PER_N:
        raise RuntimeError(
            f"N={N}: expected {BENCH_PER_N} configurations."
        )

    # Exclude topology construction: steady-state repeated-query benchmark.
    topo(N)

    exact_config=[]
    neural_config=[]

    for cid,x in enumerate(xs):
        f=RUNTIME_CACHE/f"N{N:03d}_config{cid:02d}.pkl"
        r=load_pickle(f) if f.exists() else None

        valid=(
            r is not None
            and len(r.get("exact_times",[]))==EXACT_REPEATS
            and len(r.get("neural_times",[]))==NEURAL_BLOCKS
        )

        if not valid:
            exact_full(*x)

            for _ in range(10):
                neural_query(
                    hmax,tmax,x
                )

            exact_times=[]

            for _ in range(EXACT_REPEATS):
                t0=time.perf_counter()
                exact_full(*x)
                exact_times.append(
                    time.perf_counter()-t0
                )

            neural_times=[]

            for _ in range(NEURAL_BLOCKS):
                t0=time.perf_counter()

                for _ in range(NEURAL_REPEATS):
                    neural_query(
                        hmax,tmax,x
                    )

                neural_times.append(
                    (
                        time.perf_counter()-t0
                    )
                    /
                    NEURAL_REPEATS
                )

            r={
                "N":N,
                "config_id":cid,
                "x":x,
                "exact_times":np.asarray(exact_times),
                "neural_times":np.asarray(neural_times)
            }

            atomic_pickle(r,f)

        exact_config.append(
            np.median(
                r["exact_times"]
            )
        )

        neural_config.append(
            np.median(
                r["neural_times"]
            )
        )

    exact_config=np.asarray(exact_config)
    neural_config=np.asarray(neural_config)

    runtime_detail[N]={
        "exact":exact_config,
        "neural":neural_config
    }

    emed=float(np.median(exact_config))
    fmed=float(np.median(neural_config))

    lo,hi=speedup_boot(
        exact_config,
        neural_config,
        seed=cfg.seed+N
    )

    runtime_rows.append({
        "N":N,

        "exact_sec":emed,
        "exact_q1":float(np.quantile(exact_config,.25)),
        "exact_q3":float(np.quantile(exact_config,.75)),

        "neural_sec":fmed,
        "neural_q1":float(np.quantile(neural_config,.25)),
        "neural_q3":float(np.quantile(neural_config,.75)),

        "speedup":emed/fmed,
        "speedup_CI_low":lo,
        "speedup_CI_high":hi,

        "n_config":BENCH_PER_N
    })

    print(
        f"N={N:3d} | "
        f"exact={emed:.4g}s | "
        f"neural={fmed:.4g}s | "
        f"speedup={emed/fmed:.1f}x"
    )


runtime=pd.DataFrame(runtime_rows)

runtime.to_csv(
    OUT/"runtime_full.csv",
    index=False
)


# =====================================================================================
# 20. CONTROLLED TEACHER COST
#
# Serial-equivalent exact teacher cost:
#
# C_teach(R) = sum_N n_R(N) * median[c_E(N)].
#
# This avoids mixing concurrent target-generation timings with single-query latency.
# =====================================================================================

exact_runtime_by_N={
    int(z.N):float(z.exact_sec)
    for _,z in runtime.iterrows()
}


def teacher_cost(R):
    subset=train[:R]

    counts={
        N:sum(r.N==N for r in subset)
        for N in cfg.trainN
    }

    return float(sum(
        counts[N]*exact_runtime_by_N[N]
        for N in cfg.trainN
    ))


cost_rows=[]

for R in R_GRID:
    tr=training[
        training.R==R
    ].training_sec.to_numpy()

    train_med=float(np.median(tr))
    train_q1=float(np.quantile(tr,.25))
    train_q3=float(np.quantile(tr,.75))

    Cteach=teacher_cost(R)

    cost_rows.append({
        "R":R,
        "Cteach_sec":Cteach,
        "training_sec_median":train_med,
        "training_sec_q1":train_q1,
        "training_sec_q3":train_q3,
        "offline_sec":Cteach+train_med
    })


cost=pd.DataFrame(cost_rows)


# =====================================================================================
# 21. ACCURACY THRESHOLD SELECTION
# =====================================================================================

threshold_rows=[]

for eps in EPS:
    point=None
    conservative=None

    for R in R_GRID:
        z=A(R,"Erho")

        if (
            point is None
            and z["median"]<=eps
        ):
            point=R

        if (
            conservative is None
            and z["CI_high"]<=eps
        ):
            conservative=R

    threshold_rows.append({
        "epsilon":eps,
        "R_epsilon":point,
        "R_epsilon_conservative":conservative
    })


thresholds=pd.DataFrame(threshold_rows)

thresholds.to_csv(
    OUT/"threshold_selection.csv",
    index=False
)

print("\nAccuracy thresholds")
print(thresholds.to_string(index=False))


# =====================================================================================
# 22. BOOTSTRAP BREAK-EVEN UNCERTAINTY
#
# Resamples:
#   - independent training replications for training cost
#   - independent benchmark configurations for teacher cost
#   - paired exact/neural benchmark configurations at deployment N
# =====================================================================================

def bootstrap_teacher_cost(R,B,seed):
    rng=np.random.default_rng(seed)

    subset=train[:R]

    counts={
        N:sum(r.N==N for r in subset)
        for N in cfg.trainN
    }

    out=np.zeros(B)

    for N in cfg.trainN:
        x=runtime_detail[N]["exact"]

        vals=np.empty(B)

        for b in range(B):
            idx=rng.integers(
                0,len(x),
                size=len(x)
            )

            vals[b]=np.median(
                x[idx]
            )

        out+=counts[N]*vals

    return out


def bootstrap_training_cost(R,B,seed):
    x=training[
        training.R==R
    ].training_sec.to_numpy()

    rng=np.random.default_rng(seed)
    out=np.empty(B)

    for b in range(B):
        idx=rng.integers(
            0,len(x),
            size=len(x)
        )

        out[b]=np.median(
            x[idx]
        )

    return out


def bootstrap_query_difference(N,B,seed):
    e=runtime_detail[N]["exact"]
    f=runtime_detail[N]["neural"]

    rng=np.random.default_rng(seed)
    out=np.empty(B)

    for b in range(B):
        idx=rng.integers(
            0,len(e),
            size=len(e)
        )

        out[b]=(
            np.median(e[idx])
            -
            np.median(f[idx])
        )

    return out


break_rows=[]

for _,thr in thresholds.iterrows():
    eps=float(thr["epsilon"])

    if pd.isna(thr["R_epsilon"]):
        continue

    R_eps=int(thr["R_epsilon"])

    Cteach_point=float(
        cost[
            cost.R==R_eps
        ].iloc[0].Cteach_sec
    )

    train_point=float(
        cost[
            cost.R==R_eps
        ].iloc[0].training_sec_median
    )

    offline_point=Cteach_point+train_point

    teach_boot=bootstrap_teacher_cost(
        R_eps,
        BOOT_B,
        cfg.seed+100000+R_eps
    )

    train_boot=bootstrap_training_cost(
        R_eps,
        BOOT_B,
        cfg.seed+200000+R_eps
    )

    offline_boot=teach_boot+train_boot

    for N in BENCH_N:
        rr=runtime[
            runtime.N==N
        ].iloc[0]

        diff_point=(
            float(rr.exact_sec)
            -
            float(rr.neural_sec)
        )

        if diff_point<=0:
            continue

        B_point=(
            offline_point
            /
            diff_point
        )

        diff_boot=bootstrap_query_difference(
            N,
            BOOT_B,
            cfg.seed+300000+N+R_eps
        )

        ok=diff_boot>0

        B_boot=(
            offline_boot[ok]
            /
            diff_boot[ok]
        )

        break_rows.append({
            "epsilon":eps,
            "R_epsilon":R_eps,
            "N":N,

            "Cteach_sec":Cteach_point,
            "training_sec":train_point,
            "offline_sec":offline_point,

            "exact_sec":float(rr.exact_sec),
            "neural_sec":float(rr.neural_sec),

            "B_star":B_point,
            "B_star_CI_low":float(
                np.quantile(B_boot,.025)
            ),
            "B_star_CI_high":float(
                np.quantile(B_boot,.975)
            )
        })


br=pd.DataFrame(break_rows)

br.to_csv(
    OUT/"break_even_full.csv",
    index=False
)


# =====================================================================================
# 23. MAIN TABLE
# =====================================================================================

def fmt(z):
    return (
        f"{z['median']:.4g} "
        f"[{z['CI_low']:.4g}, {z['CI_high']:.4g}]"
    )


main_rows=[]

for R in R_GRID:
    c=cost[
        cost.R==R
    ].iloc[0]

    main_rows.append({
        "R":R,

        "E2 median [95% CI]":
            fmt(A(R,"E2")),

        "Erho median [95% CI]":
            fmt(A(R,"Erho")),

        "Rel E(tau)":
            fmt(A(R,"mean_tau_rel")),

        "Rel Var(tau)":
            fmt(A(R,"var_tau_rel")),

        "Cteach (s)":
            f"{c.Cteach_sec:.0f}",

        "Training (s)":
            f"{c.training_sec_median:.0f}",

        "Offline (s)":
            f"{c.offline_sec:.0f}"
    })


main_table=pd.DataFrame(main_rows)

main_table.to_csv(
    OUT/"main_table_5_3C.csv",
    index=False
)

(
    OUT/"main_table_5_3C.tex"
).write_text(
    main_table.to_latex(
        index=False,
        escape=False
    )
)

print("\n"+"="*120)
print("MAIN TABLE")
print("="*120)
print(main_table.to_string(index=False))


# =====================================================================================
# 24. MAIN 2x2 FIGURE
#
# A. Accuracy needed to determine R_epsilon
# B. Exact versus neural complete-query runtime
# C. Speedup
# D. Accuracy-dependent break-even
# =====================================================================================

plt.rcParams.update({
    "font.size":10.5,
    "axes.spines.top":False,
    "axes.spines.right":False
})

fig,ax=plt.subplots(
    2,2,
    figsize=(11.5,8.2)
)


# -------------------------------------------------------------------------
# A. Accuracy curve
# -------------------------------------------------------------------------

med=[]
lo=[]
hi=[]

for R in R_GRID:
    z=A(R,"Erho")
    med.append(z["median"])
    lo.append(z["CI_low"])
    hi.append(z["CI_high"])

med=np.asarray(med)
lo=np.asarray(lo)
hi=np.asarray(hi)

ax[0,0].errorbar(
    R_GRID,
    med,
    yerr=np.vstack([
        med-lo,
        hi-med
    ]),
    fmt="o-",
    color="#0072B2",
    lw=2,
    capsize=3
)

for eps,style in zip(
    EPS,
    ("--",":","-.")
):
    ax[0,0].axhline(
        eps,
        color=".45",
        ls=style,
        lw=1
    )

    ax[0,0].text(
        R_GRID[0],
        eps*1.04,
        rf"$\varepsilon={eps:g}$",
        fontsize=8
    )

ax[0,0].set_xscale("log")
ax[0,0].set_yscale("log")
ax[0,0].set_xticks(R_GRID)
ax[0,0].set_xticklabels(
    ["500","1k","2k","5k","10k"]
)

ax[0,0].set_xlabel(
    r"Exact training configurations $R$"
)
ax[0,0].set_ylabel(
    r"Median $E_\rho$"
)
ax[0,0].set_title(
    "(A) Accuracy requirement"
)
ax[0,0].grid(alpha=.15)


# -------------------------------------------------------------------------
# B. Runtime
# -------------------------------------------------------------------------

rplot=runtime[
    runtime.N.isin(BENCH_N)
].sort_values("N")

x=rplot.N.to_numpy()

for y,q1,q3,label,color,marker in (
    (
        "exact_sec","exact_q1","exact_q3",
        "Exact","#D55E00","o"
    ),
    (
        "neural_sec","neural_q1","neural_q3",
        "Neural","#0072B2","s"
    )
):
    m=rplot[y].to_numpy()

    err=np.vstack([
        m-rplot[q1].to_numpy(),
        rplot[q3].to_numpy()-m
    ])

    ax[0,1].errorbar(
        x,m,
        yerr=err,
        fmt=marker+"-",
        color=color,
        lw=2,
        capsize=3,
        label=label
    )

ax[0,1].set_yscale("log")
ax[0,1].set_xlabel(
    "Population size $N$"
)
ax[0,1].set_ylabel(
    "Complete-query runtime (s)"
)
ax[0,1].set_title(
    "(B) Query runtime"
)
ax[0,1].grid(alpha=.15)
ax[0,1].legend(frameon=False)


# -------------------------------------------------------------------------
# C. Speedup
# -------------------------------------------------------------------------

speed=rplot.speedup.to_numpy()

err=np.vstack([
    speed-rplot.speedup_CI_low.to_numpy(),
    rplot.speedup_CI_high.to_numpy()-speed
])

ax[1,0].errorbar(
    x,
    speed,
    yerr=err,
    fmt="o-",
    color="#009E73",
    lw=2,
    capsize=3
)

ax[1,0].set_yscale("log")
ax[1,0].set_xlabel(
    "Population size $N$"
)
ax[1,0].set_ylabel(
    "Exact / neural runtime"
)
ax[1,0].set_title(
    "(C) Query-time speedup"
)
ax[1,0].grid(alpha=.15)


# -------------------------------------------------------------------------
# D. Break-even
# -------------------------------------------------------------------------

if len(br):
    for eps,z in br.groupby("epsilon"):
        z=z.sort_values("N")

        ax[1,1].plot(
            z.N,
            z.B_star,
            "o-",
            lw=2,
            label=rf"$\varepsilon={eps:g}$"
        )

        ax[1,1].fill_between(
            z.N.to_numpy(),
            z.B_star_CI_low.to_numpy(),
            z.B_star_CI_high.to_numpy(),
            alpha=.10
        )

ax[1,1].set_yscale("log")
ax[1,1].set_xlabel(
    "Population size $N$"
)
ax[1,1].set_ylabel(
    r"$B_\varepsilon^\star(N)$"
)
ax[1,1].set_title(
    "(D) Accuracy-dependent break-even"
)
ax[1,1].grid(alpha=.15)
ax[1,1].legend(frameon=False)


fig.suptitle(
    "Accuracy-Constrained Computational Amortization",
    fontsize=14
)

plt.tight_layout()

plt.savefig(
    OUT/"figure_5_3C_main.pdf",
    bbox_inches="tight"
)

plt.savefig(
    OUT/"figure_5_3C_main.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# =====================================================================================
# 25. CAPTION + FINAL OBJECT
# =====================================================================================

caption=r"""
Accuracy-constrained computational amortization of the exact-teacher neural
emulator. Panel (A) shows the median tail-risk error as the number R of exact
training configurations increases; error bars are hierarchical bootstrap 95%
confidence intervals over independent training replications and exact test
configurations. Panel (B) compares steady-state complete-query CPU runtimes
for the exact Markovian calculation and neural emulator, with interquartile
ranges across independent epidemic configurations. Panel (C) reports the
corresponding exact-to-neural speedup with paired configuration-bootstrap 95%
confidence intervals. Panel (D) gives the number of downstream evaluations
required to amortize teacher construction and neural training at each accuracy
requirement. The teacher cost is estimated from controlled exact-query
benchmarks at the training population sizes, rather than from concurrently
generated target timings.
""".strip()

(
    OUT/"figure_5_3C_caption.txt"
).write_text(caption)


atomic_pickle(
    {
        "scientific_config":SCIENTIFIC_CONFIG,
        "signature":SIG,
        "accuracy":accuracy,
        "training":training,
        "runtime":runtime,
        "cost":cost,
        "thresholds":thresholds,
        "break_even":br,
        "main_table":main_table
    },
    OUT/"section_5_3C_final_results.pkl"
)


# =====================================================================================
# 26. FINAL AUDIT
# =====================================================================================

print("\n"+"="*105)
print("FINAL AUDIT")
print("="*105)

for R in R_GRID:
    c=cost[cost.R==R].iloc[0]

    if (
        not np.isfinite(c.training_sec_median)
        or c.training_sec_median<=0
    ):
        raise RuntimeError(
            f"Invalid training cost at R={R}."
        )

    if not np.isclose(
        c.offline_sec,
        c.Cteach_sec+c.training_sec_median
    ):
        raise RuntimeError(
            f"Offline-cost mismatch at R={R}."
        )

    print(
        f"R={R:5d} | "
        f"Cteach={c.Cteach_sec:10.1f}s | "
        f"training={c.training_sec_median:9.1f}s | "
        f"offline={c.offline_sec:10.1f}s | OK"
    )


# =====================================================================================
# 27. STATUS
# =====================================================================================

print("\n"+"="*105)
print("5.3-C COMPLETE")
print("="*105)

print("Run:",ROOT)
print("R grid:",R_GRID)
print("Training replications:",N_REP)
print("Validation configurations:",N_VAL)
print("Test configurations:",N_TEST)
print("Runtime configurations/N:",BENCH_PER_N)
print("Bootstrap replicates:",BOOT_B)

print("\nMain table:")
print(OUT/"main_table_5_3C.tex")

print("\nMain figure:")
print(OUT/"figure_5_3C_main.pdf")

print("\nDiagnostics:")
print(OUT/"accuracy_full.csv")
print(OUT/"runtime_full.csv")
print(OUT/"break_even_full.csv")
print(OUT/"threshold_selection.csv")
print(OUT/"training_diagnostics.csv")

print(
    "\nDisconnect -> rerun -> choose 1 = RESUME.\n"
    "Scientific code changed -> increment CODE_VERSION -> "
    "choose 2 = START NEW."
)

print("="*105)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

1 = RESUME latest run
2 = START NEW run
Choose 1 or 2: 2
ROOT: /content/drive/MyDrive/StatisticalLearning/Experiment_5_3C_JASA/run_20260823_084728
Signature: ea2ea49f4da62dc7
CPU=2 | exact workers=2 | Torch threads=2
